In [3]:
import paddle
from paddleocr import PaddleOCR
from langchain_ollama import ChatOllama

print(paddle.__version__)

3.2.2


In [4]:
path1 = '../tests/invoice-1.png'
path2 = '../tests/invoice-2.jpeg'

ocr = PaddleOCR(
    text_detection_model_name="PP-OCRv5_mobile_det",
    text_recognition_model_name="PP-OCRv5_mobile_rec",
    use_doc_orientation_classify=False,
    use_doc_unwarping=False,
    use_textline_orientation=False,
)

llm = ChatOllama(
    model="gemma4:31b-cloud",
    base_url="http://localhost:11434",
    temperature=0.1,
)

# structured_llm = llm.with_structured_output(
#     InvoiceData,
#     method="json_schema",
# )

sys_message = """
You are an invoice information extraction engine.

Extract ONLY these fields:
- invoice_number
- invoice_date
- total_amount

Rules:
1. invoice_number: Extract the invoice identifier.
2. invoice_date: Extract the invoice date as dd/mm/yyyy"
3. total_amount: Extract the final amount due or total payable.
4. Do not use the subtotal, individual taxes, or line-item amounts
   as the total_amount.
5. If a field is missing, return null.
6. Never invent or calculate missing invoice information.
7. Do not extract additional fields.
8. Follow the provided structured output schema.
"""

Creating model: ('PP-OCRv5_mobile_det', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\Ruxie\.paddlex\official_models\PP-OCRv5_mobile_det`.
Creating model: ('PP-OCRv5_mobile_rec', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\Ruxie\.paddlex\official_models\PP-OCRv5_mobile_rec`.


In [5]:
def ocr_extract(path):
    result = ocr.predict(path)
    return '\n'.join(result[0]['rec_texts'])

In [6]:
res1 = ocr_extract(path1)
res2 = ocr_extract(path2)

In [7]:
def extract_invoice_data(ocr_text, sys_message):
    user_message = f"""
        Extract the invoice data from the following document.

        <Document>
        {ocr_text}
        </Document>
    """

    message = [
        ('system', sys_message),
        ('human', user_message)
    ]
    # response = structured_llm.invoke(message) structured output parser
    response = llm.invoke(message)
    return response

In [8]:
extract1 = extract_invoice_data(res1, sys_message)
extract2 = extract_invoice_data(res2, sys_message)

In [9]:
extract1

AIMessage(content='```json\n{\n  "invoice_number": "2022435",\n  "invoice_date": "19/07/2022",\n  "total_amount": "$2,510.00"\n}\n```', additional_kwargs={}, response_metadata={'model': 'gemma4:31b', 'created_at': '2026-10-24T15:42:36.510088592Z', 'done': True, 'done_reason': 'stop', 'total_duration': 310673538, 'load_duration': None, 'prompt_eval_count': 508, 'prompt_eval_duration': None, 'eval_count': 61, 'eval_duration': None, 'logprobs': None, 'model_name': 'gemma4:31b', 'model_provider': 'ollama'}, id='lc_run--01a16e94-375c-702e-af25-04005696a59b-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 508, 'output_tokens': 61, 'total_tokens': 569})

In [10]:
extract2

AIMessage(content='```json\n{\n  "invoice_number": "#dummy",\n  "invoice_date": "13/07/2023",\n  "total_amount": "$1.05"\n}\n```', additional_kwargs={}, response_metadata={'model': 'gemma4:31b', 'created_at': '2026-10-24T15:42:38.973231225Z', 'done': True, 'done_reason': 'stop', 'total_duration': 1469498641, 'load_duration': None, 'prompt_eval_count': 592, 'prompt_eval_duration': None, 'eval_count': 51, 'eval_duration': None, 'logprobs': None, 'model_name': 'gemma4:31b', 'model_provider': 'ollama'}, id='lc_run--01a16e94-425f-71aa-a1ae-04e3522c6f07-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 592, 'output_tokens': 51, 'total_tokens': 643})